# P₁ 検証実験 v3: Mistral-7B-v0.1 (32層)

## 目的
Paper II の予測 — E(l) (注意エントロピー) と Θ(l) (層間非類似度) の相関 — を 32層モデルで検証。

## 指標
1. **RBF-CKA**: σ = 中央値ヒューリスティクス
2. **Procrustes距離**: scipy.spatial.procrustes
3. **Angular距離**: コサイン類似度ベース
4. **注意エントロピー E(l)**: GQA 対応 (全ヘッド平均)

## 条件
- ランタイム: **L4 GPU** (24GB VRAM)
- モデル: mistralai/Mistral-7B-v0.1 (fp16, ~14GB VRAM)
- 入力: 20文 (多ドメイン), max_length=128

## 判定基準
- **支持**: |ρ| ≥ 0.5 かつ p < 0.05 (少なくとも1指標)
- **棄却**: 全指標で |ρ| < 0.5

In [ ]:
# セル 1: 依存パッケージのインストール
!pip install -q transformers accelerate scipy

In [ ]:
# セル 2: GPU 確認
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → L4 GPU")

In [ ]:
# セル 3: 類似度指標
import numpy as np
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.spatial import procrustes as scipy_procrustes


def rbf_cka(X: np.ndarray, Y: np.ndarray, sigma: float = None) -> float:
    """RBF カーネル CKA。X: (n, p), Y: (n, q)"""
    n = X.shape[0]
    if sigma is None:
        dists_X = pdist(X, metric='euclidean')
        dists_Y = pdist(Y, metric='euclidean')
        sigma = np.median(np.concatenate([dists_X, dists_Y]))
        if sigma < 1e-10:
            sigma = 1.0
    K = np.exp(-squareform(pdist(X, 'sqeuclidean')) / (2 * sigma ** 2))
    L = np.exp(-squareform(pdist(Y, 'sqeuclidean')) / (2 * sigma ** 2))
    H = np.eye(n) - np.ones((n, n)) / n
    Kc = H @ K @ H
    Lc = H @ L @ H
    hsic_kl = np.trace(Kc @ Lc) / (n - 1) ** 2
    hsic_kk = np.trace(Kc @ Kc) / (n - 1) ** 2
    hsic_ll = np.trace(Lc @ Lc) / (n - 1) ** 2
    denom = np.sqrt(hsic_kk * hsic_ll)
    if denom < 1e-10:
        return 0.0
    return float(np.clip(hsic_kl / denom, -1.0, 1.0))


def procrustes_distance(X: np.ndarray, Y: np.ndarray) -> float:
    """Procrustes 距離 (scipy ベース)"""
    if X.shape[1] != Y.shape[1]:
        max_d = max(X.shape[1], Y.shape[1])
        X_pad = np.zeros((X.shape[0], max_d))
        Y_pad = np.zeros((Y.shape[0], max_d))
        X_pad[:, :X.shape[1]] = X
        Y_pad[:, :Y.shape[1]] = Y
        X, Y = X_pad, Y_pad
    try:
        _, _, disparity = scipy_procrustes(X, Y)
        return float(disparity)
    except ValueError:
        return 1.0


def angular_distance(X: np.ndarray, Y: np.ndarray) -> float:
    """Angular 距離: コサイン類似度ベース"""
    X = X - X.mean(axis=0, keepdims=True)
    Y = Y - Y.mean(axis=0, keepdims=True)
    X_norms = np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-10, None)
    Y_norms = np.clip(np.linalg.norm(Y, axis=1, keepdims=True), 1e-10, None)
    X_unit = X / X_norms
    Y_unit = Y / Y_norms
    cos_sims = np.sum(X_unit * Y_unit, axis=1)
    return float(1.0 - np.mean(cos_sims))


def layer_attention_entropy(layer_attn: np.ndarray) -> float:
    """層の注意エントロピー (GQA 対応)。layer_attn: (n_heads, seq_len, seq_len)"""
    n_heads = layer_attn.shape[0]
    entropies = []
    for h in range(n_heads):
        attn = np.clip(layer_attn[h], 1e-12, None)
        row_entropy = -np.sum(attn * np.log2(attn), axis=-1)
        entropies.append(float(np.mean(row_entropy)))
    return float(np.mean(entropies))


print("✅ 指標関数定義完了")

In [ ]:
# セル 4: モデル読み込み
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
MAX_LENGTH = 128
N_SENTENCES = 20

print(f"[1/5] モデル読み込み中: {MODEL_NAME}")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    output_attentions=True,
    output_hidden_states=True,
    attn_implementation="eager",
)
model.eval()

config = model.config
n_layers = config.num_hidden_layers
n_kv_heads = getattr(config, 'num_key_value_heads', config.num_attention_heads)
n_heads = config.num_attention_heads

print(f"  層数: {n_layers}, Attention heads: {n_heads}, KV heads: {n_kv_heads}")
print(f"  Hidden dim: {config.hidden_size}")
print(f"  読み込み時間: {time.time() - t0:.1f}秒")
print(f"  VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# セル 5: 入力テキスト + 層別測定
texts = [
    # AI/ML (5文)
    "The history of artificial intelligence began in antiquity, with myths and stories of artificial beings endowed with intelligence.",
    "Machine learning is a subset of artificial intelligence that provides systems the ability to automatically learn and improve from experience.",
    "Natural language processing is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language.",
    "The Transformer architecture was introduced in the paper Attention Is All You Need published in 2017 by researchers at Google.",
    "Large language models are neural network models that are trained on large amounts of text data and can generate human-like text.",
    # 自然科学 (5文)
    "The theory of general relativity describes gravity as a geometric property of space and time, or four-dimensional spacetime.",
    "Photosynthesis is a process used by plants and other organisms to convert light energy into chemical energy that can be stored and later released.",
    "The human genome contains approximately three billion base pairs of DNA arranged into twenty-three pairs of chromosomes.",
    "Quantum entanglement is a phenomenon in which quantum states of two or more objects are interconnected regardless of the distance between them.",
    "The second law of thermodynamics states that the total entropy of an isolated system can never decrease over time.",
    # 歴史・社会 (5文)
    "The French Revolution was a period of radical political and societal change in France that began with the Estates General of 1789.",
    "The invention of the printing press by Johannes Gutenberg around 1440 revolutionized communication and the spread of knowledge across Europe.",
    "The Industrial Revolution marked a major turning point in history, fundamentally changing how people lived and worked.",
    "Ancient Greek democracy originated in the city-state of Athens around the fifth century BC and influenced political systems worldwide.",
    "The Silk Road was an ancient network of trade routes that connected the East and West from China to the Mediterranean Sea.",
    # 日常・具体 (3文)
    "The recipe for a classic French omelette requires three eggs, a tablespoon of butter, and a pinch of salt.",
    "Regular exercise has been shown to reduce the risk of chronic diseases, improve mental health, and increase life expectancy.",
    "Tokyo is one of the most densely populated cities in the world with a metropolitan population exceeding thirteen million people.",
    # 哲学・抽象 (2文)
    "The free energy principle suggests that all adaptive systems minimize a quantity called variational free energy.",
    "Information geometry studies probability distributions using concepts from differential geometry such as the Fisher information metric.",
]
texts = texts[:N_SENTENCES]

print(f"[2/5] 入力テキスト: {len(texts)}文, max_length={MAX_LENGTH}")
print(f"[3/5] 層別測定実行中...")

all_layer_entropies = [[] for _ in range(n_layers)]
all_hidden_states = [[] for _ in range(n_layers + 1)]

for i, text in enumerate(texts):
    print(f"  文 {i+1}/{len(texts)}: {text[:50]}...")
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    attentions = outputs.attentions
    hidden_states = outputs.hidden_states
    
    for l in range(n_layers):
        attn = attentions[l][0].cpu().float().numpy()
        e = layer_attention_entropy(attn)
        all_layer_entropies[l].append(e)
    
    for l in range(n_layers + 1):
        hs = hidden_states[l][0].cpu().float().numpy()
        all_hidden_states[l].append(hs)
    
    del outputs, attentions, hidden_states
    torch.cuda.empty_cache()

print("✅ 層別測定完了")

In [ ]:
# セル 6: 統計量計算 + 相関分析
print("[4/5] 統計量計算中...")

E_l = np.array([np.mean(all_layer_entropies[l]) for l in range(n_layers)])
E_l_std = np.array([np.std(all_layer_entropies[l]) for l in range(n_layers)])

rbf_cka_vals = []
procrustes_vals = []
angular_vals = []

for l in range(n_layers):
    rbf_per_sent = []
    proc_per_sent = []
    ang_per_sent = []
    for i in range(len(texts)):
        X = all_hidden_states[l][i]
        Y = all_hidden_states[l + 1][i]
        rbf_per_sent.append(rbf_cka(X, Y))
        proc_per_sent.append(procrustes_distance(X, Y))
        ang_per_sent.append(angular_distance(X, Y))
    rbf_cka_vals.append(np.mean(rbf_per_sent))
    procrustes_vals.append(np.mean(proc_per_sent))
    angular_vals.append(np.mean(ang_per_sent))
    if (l + 1) % 8 == 0:
        print(f"  層 {l+1}/{n_layers} 完了")

rbf_arr = np.array(rbf_cka_vals)
proc_arr = np.array(procrustes_vals)
ang_arr = np.array(angular_vals)
Theta_RBF = 1.0 - rbf_arr
Theta_Proc = proc_arr
Theta_Ang = ang_arr

# 相関分析
correlations = {}
for name, theta_arr in [("Θ_RBF", Theta_RBF), ("Θ_Proc", Theta_Proc), ("Θ_Ang", Theta_Ang)]:
    rho, p = stats.spearmanr(E_l, theta_arr)
    pr, pp = stats.pearsonr(E_l, theta_arr)
    correlations[name] = {
        "spearman_rho": round(float(rho), 4),
        "spearman_p": round(float(p), 6),
        "pearson_r": round(float(pr), 4),
        "pearson_p": round(float(pp), 6),
    }

print(f"\n{'='*70}")
print(f"結果 ({MODEL_NAME}, {n_layers}層)")
print(f"{'='*70}")
print(f"\n層別データ:")
print(f"{'層':>4} {'E(l)':>8} {'RBF-CKA':>9} {'Θ_RBF':>8} {'ProcDist':>9} {'AngDist':>8}")
print("-" * 55)
for l in range(n_layers):
    print(f"{l:>4d} {E_l[l]:>8.4f} {rbf_arr[l]:>9.4f} {Theta_RBF[l]:>8.4f} "
          f"{proc_arr[l]:>9.6f} {ang_arr[l]:>8.4f}")

print(f"\n相関分析: E(l) × 各 Θ 指標")
print(f"{'指標':>10} {'Spearman ρ':>12} {'p値':>12} {'Pearson r':>12} {'p値':>12}")
print("-" * 60)
for name, c in correlations.items():
    print(f"{name:>10} {c['spearman_rho']:>12.4f} {c['spearman_p']:>12.6f} "
          f"{c['pearson_r']:>12.4f} {c['pearson_p']:>12.6f}")

# 漏斗パターン
tau, tau_p = stats.kendalltau(np.arange(n_layers), E_l)

# 飽和チェック
print(f"\n━━━ 飽和問題の解消度 ━━━")
for name, arr in [("Θ_RBF", Theta_RBF), ("Θ_Proc", Theta_Proc), ("Θ_Ang", Theta_Ang)]:
    cv = np.std(arr) / (np.mean(arr) + 1e-10)
    n_nonzero = np.sum(arr > 0.001)
    print(f"  {name}: mean={np.mean(arr):.4f}, sd={np.std(arr):.4f}, "
          f"CV={cv:.2f}, 有効点={n_nonzero}/{n_layers}")

# 中間層相関
mid = slice(2, n_layers - 2)
print(f"\n━━━ 中間層 (2-{n_layers-3}) の相関 ━━━")
for name, theta_arr in [("Θ_RBF", Theta_RBF), ("Θ_Proc", Theta_Proc), ("Θ_Ang", Theta_Ang)]:
    rho_m, p_m = stats.spearmanr(E_l[mid], theta_arr[mid])
    pr_m, pp_m = stats.pearsonr(E_l[mid], theta_arr[mid])
    print(f"  {name}: Spearman ρ={rho_m:+.4f} (p={p_m:.4f}), Pearson r={pr_m:+.4f} (p={pp_m:.4f})")

# 判定
any_supported = any(
    abs(c["spearman_rho"]) >= 0.5 and c["spearman_p"] < 0.05
    for c in correlations.values()
)
all_rejected = all(abs(c["spearman_rho"]) < 0.5 for c in correlations.values())

print(f"\n━━━ 仮説判定 ━━━")
best_metric = max(correlations.items(), key=lambda x: abs(x[1]["spearman_rho"]))
print(f"  最強指標: {best_metric[0]} (ρ={best_metric[1]['spearman_rho']}, p={best_metric[1]['spearman_p']})")
print(f"  漏斗パターン: τ={tau:.4f}, p={tau_p:.6f}")

if any_supported:
    print(f"  判定: ✅ 支持 — |ρ| ≥ 0.5 (p < 0.05)")
elif all_rejected:
    print(f"  判定: ❌ 棄却 — 全指標で |ρ| < 0.5")
else:
    print(f"  判定: ⚠️ 混在")

In [ ]:
# セル 7: 結果保存 + JSON ダウンロード
import json

result = {
    "experiment": "P1_verification_v3_mistral_7b",
    "model": MODEL_NAME,
    "n_layers": n_layers,
    "n_sentences": len(texts),
    "n_heads": n_heads,
    "n_kv_heads": n_kv_heads,
    "hidden_size": config.hidden_size,
    "max_length": MAX_LENGTH,
    "layer_data": {
        "E_l": E_l.tolist(),
        "E_l_std": E_l_std.tolist(),
        "RBF_CKA": rbf_arr.tolist(),
        "Theta_RBF": Theta_RBF.tolist(),
        "Procrustes_dist": proc_arr.tolist(),
        "Angular_dist": ang_arr.tolist(),
    },
    "correlations": correlations,
    "funnel_pattern": {
        "kendall_tau": round(float(tau), 4),
        "kendall_p": round(float(tau_p), 6),
    },
    "saturation_resolved": {
        name: {
            "mean": round(float(np.mean(arr)), 4),
            "std": round(float(np.std(arr)), 4),
            "cv": round(float(np.std(arr) / (np.mean(arr) + 1e-10)), 4),
            "n_nonzero": int(np.sum(arr > 0.001)),
        }
        for name, arr in [("Theta_RBF", Theta_RBF), ("Theta_Proc", Theta_Proc), ("Theta_Ang", Theta_Ang)]
    },
    "hypothesis_judgment": {
        "any_supported": bool(any_supported),
        "all_rejected": bool(all_rejected),
        "best_metric": best_metric[0],
        "best_rho": best_metric[1]["spearman_rho"],
        "best_p": best_metric[1]["spearman_p"],
    },
}

outpath = "pei_p1_mistral_7b_results.json"
with open(outpath, "w") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print(f"✅ 結果を {outpath} に保存")

# Colab からダウンロード
try:
    from google.colab import files
    files.download(outpath)
    print("📥 ダウンロード開始")
except ImportError:
    print("ローカル実行: ダウンロードスキップ")

In [ ]:
# セル 8: 可視化 (オプション)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'P₁ Verification: {MODEL_NAME} ({n_layers} layers)', fontsize=14)

layers = np.arange(n_layers)

# E(l) プロファイル
ax = axes[0, 0]
ax.plot(layers, E_l, 'b-o', markersize=3, label='E(l)')
ax.fill_between(layers, E_l - E_l_std, E_l + E_l_std, alpha=0.2)
ax.set_xlabel('Layer')
ax.set_ylabel('Attention Entropy E(l)')
ax.set_title('Attention Entropy Profile')
ax.grid(True, alpha=0.3)

# Θ プロファイル (3指標)
ax = axes[0, 1]
ax.plot(layers, Theta_RBF, 'r-o', markersize=3, label='Θ_RBF')
ax.plot(layers, Theta_Proc, 'g-s', markersize=3, label='Θ_Proc')
ax.plot(layers, Theta_Ang, 'm-^', markersize=3, label='Θ_Ang')
ax.set_xlabel('Layer')
ax.set_ylabel('Dissimilarity Θ(l)')
ax.set_title('Layer Dissimilarity Profile')
ax.legend()
ax.grid(True, alpha=0.3)

# E(l) vs Θ_RBF scatter
ax = axes[1, 0]
c = correlations['Θ_RBF']
ax.scatter(E_l, Theta_RBF, c=layers, cmap='viridis', s=30)
ax.set_xlabel('E(l)')
ax.set_ylabel('Θ_RBF(l)')
ax.set_title(f'E(l) vs Θ_RBF: ρ={c["spearman_rho"]:.3f}, p={c["spearman_p"]:.4f}')
ax.grid(True, alpha=0.3)

# E(l) vs Θ_Proc scatter
ax = axes[1, 1]
c2 = correlations['Θ_Proc']
ax.scatter(E_l, Theta_Proc, c=layers, cmap='viridis', s=30)
ax.set_xlabel('E(l)')
ax.set_ylabel('Θ_Proc(l)')
ax.set_title(f'E(l) vs Θ_Proc: ρ={c2["spearman_rho"]:.3f}, p={c2["spearman_p"]:.4f}')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('p1_mistral_7b_results.png', dpi=150, bbox_inches='tight')
plt.show()

try:
    from google.colab import files
    files.download('p1_mistral_7b_results.png')
except ImportError:
    pass

print("✅ 全実験完了")